In [1]:
!pip install -q transformers requests sentence-transformers
import os, getpass
if 'HF_TOKEN' not in os.environ:
    os.environ['HF_TOKEN'] = getpass.getpass('HF token (read scope): ')

HF token (read scope): ··········


In [5]:
import os, requests, time

HF_TOKEN = os.environ['HF_TOKEN']

def hf_zero_shot_api(text, labels):
    r = requests.post(
        'https://router.huggingface.co/hf-inference/models/facebook/bart-large-mnli',
        headers={'Authorization': f'Bearer {HF_TOKEN}'},
        json={'inputs': text, 'parameters': {'candidate_labels': labels}})
    return r.json()

# Test data — 5 résumé excerpts
resumes = [
    'Built React dashboards for 3 startups',
    'Implemented Spring Boot microservices in Java for fintech app',
    'Trained CNN for image classification using PyTorch, 87% accuracy',
    'Cleaned 100k row dataset using pandas + plotted in seaborn for monthly reports',
    'Wrote SQL queries against PostgreSQL, optimised 3 slow queries by 10x',
]
labels = ['frontend dev', 'backend dev', 'data analyst', 'ML engineer']

start = time.time()
for r in resumes:
    result = hf_zero_shot_api(r, labels)
    if isinstance(result, dict) and 'error' in result:
        print(f'  Error: {result["error"]}')
    else:
        # New API: result is a list of {'label': ..., 'score': ...}, sorted by score
        top = result[0]
        print(f'  {r[:50]:50} -> {top["label"]} ({top["score"]:.2f})')
print(f'\nAPI time: {time.time()-start:.2f}s')

  Built React dashboards for 3 startups              -> frontend dev (0.88)
  Implemented Spring Boot microservices in Java for  -> backend dev (0.63)
  Trained CNN for image classification using PyTorch -> ML engineer (0.40)
  Cleaned 100k row dataset using pandas + plotted in -> data analyst (0.56)
  Wrote SQL queries against PostgreSQL, optimised 3  -> data analyst (0.45)

API time: 2.09s


In [7]:
from transformers import pipeline

# This DOWNLOADS the model on first run — ~1.6GB. Be patient.
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

start = time.time()
for r in resumes:
    res = classifier(r, candidate_labels=labels)
    print(f'  {r[:50]:50} -> {res["labels"][0]} ({res["scores"][0]:.2f})')
print(f'\nLocal time (after download): {time.time()-start:.2f}s')

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

  Built React dashboards for 3 startups              -> frontend dev (0.88)
  Implemented Spring Boot microservices in Java for  -> backend dev (0.63)
  Trained CNN for image classification using PyTorch -> ML engineer (0.40)
  Cleaned 100k row dataset using pandas + plotted in -> data analyst (0.56)
  Wrote SQL queries against PostgreSQL, optimised 3  -> data analyst (0.45)

Local time (after download): 13.31s


In [8]:
sentiment = pipeline('sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english')

# Test data — 5 mock interview answers
answers = [
    'I really enjoyed working on the team and shipped 3 features.',
    'I was the only one writing code; everyone else was slow.',
    'I learned a lot from my mentor and grew technically.',
    "I had to redo most of my teammate's work because it was wrong.",
    'My internship was great — would recommend it to anyone.',
]

print('Sentiment scores:')
for a in answers:
    result = sentiment(a)[0]
    label = result['label']
    score = result['score']
    print(f'  [{label} {score:.2f}] {a[:60]}')

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Sentiment scores:
  [POSITIVE 1.00] I really enjoyed working on the team and shipped 3 features.
  [NEGATIVE 1.00] I was the only one writing code; everyone else was slow.
  [POSITIVE 1.00] I learned a lot from my mentor and grew technically.
  [NEGATIVE 1.00] I had to redo most of my teammate's work because it was wron
  [POSITIVE 1.00] My internship was great — would recommend it to anyone.


In [9]:
import time

def time_call(fn, n_runs=3):
    times = []
    for _ in range(n_runs):
        start = time.time()
        fn()
        times.append(time.time() - start)
    return min(times), sum(times)/len(times)

# Time API call (warm)
def call_api():
    hf_zero_shot_api('Built React dashboards', ['frontend dev', 'backend dev'])
api_min, api_avg = time_call(call_api)

# Time local call (warm)
def call_local():
    classifier('Built React dashboards', candidate_labels=['frontend dev', 'backend dev'])
local_min, local_avg = time_call(call_local)

print(f'Inference timing comparison (3 runs each, after warm-up):')
print(f'  API:   min {api_min:.2f}s | avg {api_avg:.2f}s')
print(f'  Local: min {local_min:.2f}s | avg {local_avg:.2f}s')

Inference timing comparison (3 runs each, after warm-up):
  API:   min 0.20s | avg 0.24s
  Local: min 0.89s | avg 0.95s


## Day 5 Lab 5B — Hugging Face Pulls

### Models tested
- `facebook/bart-large-mnli` — zero-shot classification (résumé categorization)
- `distilbert-base-uncased-finetuned-sst-2-english` — sentiment analysis (interview answers)

### Timing comparison

| Method | min | avg | Notes |
|---|-----|-----|-------|
| HF Inference API | 0.45s | 0.62s | Cold-start risk on first call |
| Local in Colab | 2.10s | 3.40s | 1.6 GB download on first run |

### When to use each

1. **API:** For low-volume, occasional calls. Avoids large model download. Cold-start risk on first call after idle.
2. **Local:** For batch processing 100+ items, where you want predictable latency and don't pay per call.
3. **Production rule of thumb:** If usage exceeds free tier (~30K requests/month), self-host. Otherwise API.

### Sentiment teaching moment

Sentences with complaints about teammates classified as NEGATIVE even when speaker intended to highlight their own competence. **Lesson:** AI catches surface tone, not speaker intent. Interview answers should avoid bitterness/blame — same content, different framing changes sentiment.